In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('numpy')
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: numpy


In [3]:
from moabb.paradigms import P300
from moabb.datasets import *
from hoda.tensorize import fh_power, fh_log_envelope, hankel_tensor
from hoda.classification import ZLogRatio, ZScore
import numpy as np
import scipy.linalg

dataset = BNCI2014_008()

paradigm = P300(resample=48)
X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[1],
)
print(X.shape)
#X = hankel_tensor(X)
X = np.apply_along_axis(scipy.linalg.hankel, -1,X)
X = tl.tensor(X)
X.shape

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>



(4200, 8, 48)


(4200, 8, 48, 48)

In [4]:
from hoda.hoda import HODA

hoda = HODA(
        rank=None,
        max_iter=64,
        tol=1e-6,
        shrinkage='lw',
        obj='tr',
        solver='lanczos',
        toeplitz=None,
        taper=(1,2),
        extra_train_info=False,
        verbose=True,
        theta=0.9,
        forward=True,
        refit_shrinkage=True,
)


In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from hoda.cov import mode_scatter
plt.style.use('default')

%load_ext line_profiler
%lprun -f mode_scatter hoda.fit_backward(X,y)
df = pd.DataFrame(hoda.train_info_['backward'])

Backward HODA model rank=(2, 30, 30): 100%|██████████| 64/64 [01:57<00:00,  1.84s/it]
/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning:

Maximum number of iterations reached without convergence



Timer unit: 1e-09 s

Total time: 67.6148 s
File: /vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/cov.py
Function: mode_scatter at line 17

Line #      Hits         Time  Per Hit   % Time  Line Contents
    17                                           def mode_scatter(
    18                                               X, k, weights=None, shrinkage=0, toeplitz=None, taper=False, assume_centered=False
    19                                           ):
    20                                               """Calculate the scatter matrix along a given tensor mode"""
    21                                           
    22       393    1771453.0   4507.5      0.0      n_samples, *shape = X.shape
    23       393     534006.0   1358.8      0.0      order = len(shape)
    24       393     139663.0    355.4      0.0      n_features = shape[k]
    25                                               # Determine mode scatter
    26       393    2108664.0   5365.6      0.0      modes = 

In [6]:
df

,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,1.012246,0.000003,3.288441e+08
1,1,2,2,1.418148,0.000499,1.375803e+09
2,1,3,3,1.402681,0.000318,2.282247e+09
3,2,1,4,1.380197,0.000006,1.137691e+08
4,2,2,5,1.095740,0.000546,1.432920e+09
...,...,...,...,...,...,...
187,63,2,188,0.003327,0.000609,1.496609e+09
188,63,3,189,0.004495,0.000626,2.098036e+09
189,64,1,190,0.000035,0.000009,4.347526e+07
190,64,2,191,0.003293,0.000609,1.497071e+09


In [7]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr')
    fig.show()


In [8]:
if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_rt')
    fig.show()


In [9]:
px.line(df, x='flip', y='update', log_y=True, color='mode')


In [10]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [11]:
px.line(df, x='flip', y='objective', log_y=True, color='mode')

In [12]:
hoda.fit_forward(X,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

Forward model : 100%|██████████| 63/63 [01:57<00:00,  1.87s/it]


,iteration,mode,flip,update,lambda_
0,1,1,1,0.836881,0.0
1,1,2,2,1.138566,0.0
2,1,3,3,1.212445,0.0
3,2,1,4,0.177957,0.0
4,2,2,5,0.824185,0.0
...,...,...,...,...,...
184,62,2,185,0.007055,0.0
185,62,3,186,0.008968,0.0
186,63,1,187,0.000047,0.0
187,63,2,188,0.006687,0.0


In [13]:
if hoda.extra_train_info:
    px.line(df, x='flip', y='mse', log_y=True)

In [14]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [15]:
from sklearn.feature_selection import SelectFpr, SelectFwe, SelectFdr
import numpy as np
from sklearn.preprocessing import StandardScaler

Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))
xt = StandardScaler().fit_transform(xt)

select = SelectFdr(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})
df

,feature,F,p_value,significant
0,0,0.276074,0.599314,False
1,1,1.143512,0.284973,False
2,2,0.057907,0.809847,False
3,3,0.000222,0.988119,False
4,4,0.018066,0.893084,False
...,...,...,...,...
1795,1795,0.154486,0.694305,False
1796,1796,2.348769,0.125457,False
1797,1797,1.506471,0.219747,False
1798,1798,3.777415,0.052016,False


In [16]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [17]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

x_viz = PCA(n_components=2).fit_transform(xts)
fig = px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)
fig.update_layout(width=1000, height=800)
fig